# Dilution or noisy quality signal? Breaking the confound

Section 5.9 holds the total corpus near 4,000 images while growing K, so
client count and per-client sample size move together: K = 16 leaves each
site 250 images. The advantage decays, but the paper cannot say whether
that is because more clients dilute the weight a corrupted site can be
denied, or because 250 images make the quality signal too noisy to act
on. The section states this and names the experiment that separates them.

This is that experiment. Per-client data is held FIXED at 1,000 images --
the same as the K = 4 experiments -- while K grows to 8, 12 and 16. The
corpus grows with the federation (8,000 / 12,000 / 16,000 images).

If the decay persists, it is dilution and 5.9's first mechanism is right.
If the advantage returns, the decay was a small-sample artifact and the
constant-total design was measuring the wrong thing.


In [ ]:
import glob, os, shutil, sys
REPO='/kaggle/working/repo'
hits=glob.glob('/kaggle/input/**/paper/build_paper.py', recursive=True)
assert hits, 'attach the aha-fed-code dataset'
src=os.path.dirname(os.path.dirname(hits[0]))
if not os.path.isdir(os.path.join(REPO,'paper')):
    shutil.copytree(src, REPO, dirs_exist_ok=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print('staged:', os.getcwd())


In [ ]:
!pip install -q medmnist opacus

In [ ]:
import subprocess, sys
PROBE=('import torch;'
       'a=torch.randn(4,3,8,8,device="cuda");'
       'w=torch.randn(2,3,3,3,device="cuda");'
       'torch.nn.functional.conv2d(a,w).sum().item();'
       'print(torch.cuda.get_device_name(0), torch.__version__)')
r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])
if r.returncode!=0:
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'torch==2.4.1','torchvision==0.19.1',
                    '--index-url','https://download.pytorch.org/whl/cu118'],check=True)
    r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
    print(r.stdout.strip() or r.stderr.strip()[-400:])
    if r.returncode!=0: raise SystemExit('no working torch/GPU combination')


In [ ]:
from paper.run_real_experiments import run_one, OUT_DIR

SEEDS=[42,123,456,7,13,21,99,2024,31337,555]

# 25% corrupted, per-client data FIXED at 1000 (the corpus grows with K).
# Contrast with the frac_* sweep, which held the corpus fixed instead.
GRID=[(8,2),(12,3),(16,4)]

for K,nn in GRID:
    for s in SEEDS:
        common=dict(dataset='pathmnist', num_classes=9,
                    num_clients=K, num_noisy=nn, noisy_client=K-1,
                    samples_per_client=1000, seed=s)
        run_one(OUT_DIR/f'fixn_fedavg_k{K}_n{nn}_s{s}.json',
                aggregator='fedavg', **common)
        run_one(OUT_DIR/f'fixn_aha_k{K}_n{nn}_s{s}.json',
                aggregator='adaptive', alpha=1.0, beta=4.0, gamma=4.0,
                **common)
    print(f'K={K}, {nn} corrupted, 1000 img/client done', flush=True)


In [ ]:
import glob, json, re, statistics as st
from pathlib import Path
from scipy.stats import wilcoxon

def by_seed(pat, key='ece'):
    out={}
    for f in sorted(glob.glob(pat)):
        d=json.loads(Path(f).read_text())
        fin=d.get('final') or {}
        m=re.search(r'_s(\d+)$', Path(f).stem)
        if fin.get(key) is not None:
            out[m.group(1) if m else '42']=fin[key]
    return out

print('25% corrupted, per-client data fixed at 1000')
print('  K    AHA-Fed   FedAvg    ratio    p        n')
fixed={}
for K,nn in [(8,2),(12,3),(16,4)]:
    a=by_seed(f'runs/paper/real/fixn_aha_k{K}_n{nn}_s*.json')
    v=by_seed(f'runs/paper/real/fixn_fedavg_k{K}_n{nn}_s*.json')
    c=sorted(set(a)&set(v))
    if not c: continue
    x=[a[k] for k in c]; y=[v[k] for k in c]
    p=wilcoxon(x,y).pvalue
    fixed[K]=st.mean(y)/st.mean(x)
    print(f'  {K:2d}   {st.mean(x):.4f}   {st.mean(y):.4f}   '
          f'{fixed[K]:.2f}x   p={p:.4f}  n={len(c)}')

print()
print('constant-total comparison (from the frac_* sweep):')
for K,nn in [(8,2),(12,3),(16,4)]:
    a=by_seed(f'runs/paper/real/frac_aha_k{K}_n{nn}_s*.json')
    v=by_seed(f'runs/paper/real/frac_fedavg_k{K}_n{nn}_s*.json')
    c=sorted(set(a)&set(v))
    if c:
        print(f'  K={K:2d}: {st.mean([v[k] for k in c])/st.mean([a[k] for k in c]):.2f}x'
              f'   (fixed-n: {fixed.get(K, float("nan")):.2f}x)')

if len(fixed)==3:
    print()
    if fixed[16] > 1.15:
        print('=> advantage RETURNS at 1000 img/client: the decay was a')
        print('   small-sample artifact, not dilution. 5.9 needs rewriting.')
    elif fixed[8] > fixed[12] > fixed[16]:
        print('=> decay PERSISTS with per-client data held fixed:')
        print('   dilution is the mechanism, and the confound is resolved.')
    else:
        print('=> mixed; report the pattern as measured.')


In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/fixn_results.zip runs/paper/real
!ls -lh /kaggle/working/fixn_results.zip
